In [ ]:
import time
import logging
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [143]:
"""
scraper.py
Extrae reseñas de Teamblind usando Selenium + BeautifulSoup.
Teamblind usa renderizado JS, por lo que se requiere Selenium.
"""
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

class TeamblindScraper:
    """
    Uso:
        scraper = TeamblindScraper(company="google", max_pages=5)
        df = scraper.run()
    """
    BASE_URL = "https://www.teamblind.com/company/{company}/reviews?page={page}"

    def __init__(self, company: str = "google", max_pages: int = 5, headless: bool = True):
        self.company = company.lower().replace(" ", "-")
        self.max_pages = max_pages
        self.headless = headless
        self.driver = None

    # ------------------------------------------------------------------
    # Driver setup
    # ------------------------------------------------------------------

    def _init_driver(self):
        options = Options()
        if self.headless:
            options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        #Simulador de navegador real
        options.add_argument("--window-size=1280,800")
        options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

        self.driver = webdriver.Chrome(options=options)
        logger.info("Driver inicializado correctamente.")

    def _close_driver(self):
        if self.driver:
            self.driver.quit()
            logger.info("Driver cerrado.")

    # ------------------------------------------------------------------
    # Scraping
    # ------------------------------------------------------------------

    def _get_page_source(self, url: str) -> str:
        self.driver.get(url)
        try:
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.review-card, article"))
            )
        except Exception:
            logger.warning(f"Timeout esperando contenido en: {url}")
        time.sleep(2)  # buffer extra para JS dinámico
        return self.driver.page_source

    def _parse_reviews(self, html: str) -> list[dict]:
        #Parsea las reseñas de una página HTML.
        soup = BeautifulSoup(html, "html.parser")         
        
        reviews = []

        # Selector de cards y propiedades
        cards = soup.find_all(
            'div',
            class_=lambda c: c and all(
                cls in c for cls in ['flex', 'w-full', 'flex-col', 'gap-4', 'bg-white']
            )
        )
        logger.info(f"Cards encontradas: {len(cards)}")

        for card in cards:
            try:
                rating_block = card.find('div', class_=lambda c: c and 'items-center' in c and 'gap-2' in c)
                rating = rating_block.find('h3', class_='font-semibold') if rating_block else None

                main_block = card.find('div', class_=lambda c: c and 'flex-1' in c)
                if not main_block:
                    continue

                title = main_block.find('h3', class_='font-semibold')

                description = main_block.find('div', class_=lambda c: c and 'text-xs' in c and 'text-gray-700' in c)
                info_description = description.get_text(strip=True) if description else ""
                parts_info = [p.strip() for p in info_description.split('•')]

                # FIX 2: parts_info ya son strings, NO usar .get_text()
                role = parts_info[2] if len(parts_info) > 2 else ""
                date = parts_info[3] if len(parts_info) > 3 else ""

                pros = main_block.find('div', class_=lambda c: c and 'mt-4' in c)
                pros_text = pros.find('p').get_text(strip=True) if pros and pros.find('p') else ""

                contras = main_block.find('div', class_=lambda c: c and 'my-4' in c)
                contras_text = contras.find('p').get_text(strip=True) if contras and contras.find('p') else ""

                # FIX 3: Concatenación correcta de pros + contras
                review_text = " ".join(filter(None, [pros_text, contras_text]))

                review = {
                    'title': title.get_text(strip=True) if title else "",
                    'review': review_text,
                    'rating': rating.get_text(strip=True) if rating else "",
                    'role': role,   # Ya es string
                    'date': date,   # Ya es string
                    'company': self.company,
                }
                # Solo guardar si tiene contenido real
            
                if review["rating"] or review["title"]:
                    reviews.append(review)

            except Exception as e:
                logger.debug(f"Error parseando card: {e}")
                continue

        return reviews

    def run(self) -> pd.DataFrame:
        """Ejecuta el scraper y retorna un DataFrame."""
        self._init_driver()
        all_reviews = []
        
        try:
            for page in range(1, self.max_pages + 1):
                url = self.BASE_URL.format(company=self.company, page=page)
                logger.info(f"Scrapeando página {page}: {url}")

                html = self._get_page_source(url)
                reviews = self._parse_reviews(html)

                if not reviews:
                    logger.info(f"Sin reseñas en página {page}. Terminando.")
                    break

                all_reviews.extend(reviews)
                logger.info(f"  → {len(reviews)} reseñas encontradas (total: {len(all_reviews)})")
                time.sleep(1.5)  # respetar el rate limiting

        finally:
            self._close_driver()

        df = pd.DataFrame(all_reviews)
        logger.info(f"Scraping completo. Total de reseñas: {len(df)}")
        return df

    # ------------------------------------------------------------------
    # Guardar datos
    # ------------------------------------------------------------------

    def save(self, df: pd.DataFrame, output_dir: str = "data/raw") -> str:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        path = f"{output_dir}/reviews_raw.csv"
        df.to_csv(path, mode='a', header=False, index=False, encoding="utf-8")
        logger.info(f"Datos guardados en: {path}")
        return path

In [144]:
companies = ['apple', 'microsoft', 'meta', 'amazon', 'openai', 'paypal', 'uber', 'salesforce', 'linkedin', 'reddit', 'netflix', 'intel', 'nvidia', 'ibm', 'oracle', 'samsung', 't-mobile','nordstrom', 'verizon', 'mongodb', 'sofi', 'tubi-tv', 'flexport', 'robinhood', 'adobe', 'sap', 'foxconn', 'tesla', 'alibaba', 'herbalife']
for c in companies:
    scraper = TeamblindScraper(
        company=c,
        max_pages=1,
        headless=True
    )
    df = scraper.run()
    scraper.save(df)

2026-05-03 12:03:58,528 [INFO] Driver inicializado correctamente.
2026-05-03 12:03:58,528 [INFO] Scrapeando página 1: https://www.teamblind.com/company/apple/reviews?page=1
2026-05-03 12:04:09,441 [WARNING] Timeout esperando contenido en: https://www.teamblind.com/company/apple/reviews?page=1
2026-05-03 12:04:11,531 [INFO] Cards encontradas: 30
2026-05-03 12:04:11,533 [INFO]   → 30 reseñas encontradas (total: 30)
2026-05-03 12:04:13,130 [INFO] Driver cerrado.
2026-05-03 12:04:13,131 [INFO] Scraping completo. Total de reseñas: 30
2026-05-03 12:04:13,132 [INFO] Datos guardados en: data/raw/reviews_raw.csv
2026-05-03 12:04:13,561 [INFO] Driver inicializado correctamente.
2026-05-03 12:04:13,562 [INFO] Scrapeando página 1: https://www.teamblind.com/company/microsoft/reviews?page=1
2026-05-03 12:04:25,282 [WARNING] Timeout esperando contenido en: https://www.teamblind.com/company/microsoft/reviews?page=1
2026-05-03 12:04:27,376 [INFO] Cards encontradas: 30
2026-05-03 12:04:27,378 [INFO]   →